In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
# Load training dataset

df = pd.read_csv("train.csv", index_col = "Id")

**Overview of training data**

In [ ]:
# Data overview

print("df shape: ", df.shape)
print("Missing values: \n", df.isnull().sum().sort_values(ascending=False))
print("Data types: \n", df.dtypes.value_counts())
df.head()


In [ ]:
# Target variable exploration

fig, axes = plt.subplots(ncols=5, figsize = (16, 4)) 

sns.histplot(df["SalePrice"], ax=axes[0])
sns.kdeplot(df["SalePrice"], ax=axes[1])
sns.boxplot(df["SalePrice"], ax=axes[2])
sns.histplot(np.log1p(df["SalePrice"]), ax=axes[3])
sns.kdeplot(np.log1p(df["SalePrice"]), ax=axes[4])

plt.tight_layout()
plt.show()

Since house prices appear normally distributed under log transform, the new target variable will be log(SalePrice), to be converted back to SalePrice after prediction.

In [ ]:
# Convert features to categorical

df["MoSold"] = df["MoSold"].astype("str")
df["MSSubClass"] = df["MSSubClass"].astype("str")

**Missing values**

In [ ]:
# Missing value analysis

missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Fill missing with None

cols_to_fill_with_NA = ["PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType", "FireplaceQu", "GarageType", "GarageQual", "GarageCond", "BsmtExposure", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtFinType1", "Electrical", "GarageFinish"]

df[cols_to_fill_with_NA] = df[cols_to_fill_with_NA].fillna("None")

In [ ]:
# Fill missing lot frontage with median for neighbourhood

df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
# Fill MasVnrArea by median of MasVnrType

df['MasVnrArea'] = df['MasVnrArea'].fillna(
    df.groupby('MasVnrType')['MasVnrArea'].transform('median')
)

In [ ]:
# Fix missing garage age by creating new features, and filling

df["has_garage"] = df["GarageYrBlt"].notnull().astype(bool)
df["garage_age"] = df["YrSold"] - df["GarageYrBlt"]
df["garage_age"] = df["garage_age"].fillna(0)

df = df.drop(columns="GarageYrBlt")

# Check working correctly, false garage should have 0 age, true garage has actual age.

df[["has_garage", "garage_age"]].sample(10)

In [ ]:
# Check missingess

df.isnull().sum().sum()

**Numeric features**

In [ ]:
# Numeric feature distrubtion

numeric_cols = df.drop(columns = "SalePrice").select_dtypes(np.number).columns

print("Numeric columns: ", len(numeric_cols))

In [ ]:
# Plot numeric feature distributions in grid

fig, axes = plt.subplots(6, 6, sharex = False, sharey = False, figsize = (20, 20))

# Make looping easier
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], bins = 40, ax = axes[i])

plt.tight_layout()
plt.show()

Many heavily skewed features - implies tree based model will be more appropriate.

Many of these features can be converted to binary e.g. has_pool.

In [ ]:
# Numeric features vs sale price

fig, axes = plt.subplots(6, 6, figsize=(20, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.scatterplot(x=df[col], y=df["SalePrice"], alpha=0.5, ax = axes[i])

plt.tight_layout()
plt.show()

Evidence of outliers. Identify using table or scatterplot and remove.

In [ ]:
# Look at GrLivArea outliers

df[["GrLivArea", "SalePrice"]].sort_values(by="GrLivArea", ascending=False).head()

In [ ]:
# Look at MiscVal outliers

df[["MiscVal", "SalePrice"]].sort_values(by="MiscVal", ascending=False).head()

In [ ]:
# Remove obvious outliers

df = df[
    (df["LotFrontage"] < 300) &
    (df["LotArea"] < 100000) &
    (df["BsmtFinSF1"] < 4000) &
    (df["TotalBsmtSF"] < 6000) &
    (df["GrLivArea"] < 4676) &
    (df["MiscVal"] < 8300) &
    (df["BedroomAbvGr"] < 8) &
    (df["LotArea"] < 70000) &
    (df["MasVnrArea"] < 1400) &
    (df["OpenPorchSF"] < 400)
]

In [ ]:
# Re check feature vs sale price after outlier removal

fig, axes = plt.subplots(6, 6, figsize=(20, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.scatterplot(x=df[col], y=df["SalePrice"], alpha=0.5, ax = axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Look at feature correlations to SalePrice

top_features = df.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)

In [ ]:
# Most correlated features vs Sale price

sns.pairplot(data=df, x_vars=top_features.index[1:8], y_vars=["SalePrice"])

This shows strongs relationships between these features and target. Next check least import features to confirm no relationship.

In [ ]:
# Find lowest correlation features and plot against sale price
low_correlation_features = top_features[np.abs(top_features) < 0.2]

sns.pairplot(data=df, x_vars=low_correlation_features.index, y_vars=["SalePrice"])

In [ ]:
# Correlation matrix for numeric features

corr = df[numeric_cols].corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr)

Many features are correlated, suggesting linear models are unsuitable without more detailed feature engineering/investigation.

**Numeric feature engineering**

In [ ]:
# Initiate columns to drop
cols_to_drop = []

Convert massively skewed features into binary where appropriate. Keep 2nd floor SF as still important feature.

In [ ]:
# Feature convert to binary

df["has_second_floor"] = df["2ndFlrSF"]

features_to_binary = ["has_second_floor", "3SsnPorch", "ScreenPorch", "PoolArea", "EnclosedPorch"]

for col in features_to_binary:
    df[col] = df[col] > 0

In [ ]:
# Create house_age

df["house_age"] = df["YrSold"] - df["YearBuilt"]

cols_to_drop.append("YrSold")
cols_to_drop.append("YearBuilt")

Remove features that clearly have no relationship with sale price or will negatively impact model performance.

In [ ]:
# Drop unpredictable features

cols_to_drop.append("BedroomAbvGr")
cols_to_drop.append("BsmtFinSF2")
cols_to_drop.append("BsmtHalfBath")
cols_to_drop.append("LowQualFinSF")
cols_to_drop.append("MiscVal")
cols_to_drop.append("KitchenAbvGr")
cols_to_drop.append("BsmtFullBath")
cols_to_drop.append("HalfBath")

In [ ]:
# Re check numeric features vs sale price after engineering

new_numeric_cols = df.drop(columns = ["SalePrice"] + cols_to_drop).select_dtypes(np.number).columns

len(new_numeric_cols)

In [ ]:
# Plot new numeric features vs sale price

fig, axes = plt.subplots(7, 3, figsize=(20, 20))
axes = axes.flatten()

for i, col in enumerate(new_numeric_cols):
    sns.scatterplot(x=df[col], y=df["SalePrice"], alpha=0.5, ax = axes[i])

plt.tight_layout()
plt.show()

**Categorical features**

In [ ]:
# Look at categorical columns

cat_cols = df.select_dtypes(include=['object', 'category', 'str']).columns

print("Categorical columns: ", len(cat_cols))


In [ ]:
# Look at cat col distributions in grid

fig, axes = plt.subplots(9, 5, sharex = False, sharey = False, figsize = (20, 20))

# Make looping easier
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(x = col, data = df, ax=axes[i])
    axes[i].tick_params(axis='x', rotation=45)
    
plt.tight_layout()
plt.show()

Some features are massively skewed, check the boxplots for the categories and ANOVA if further inspection required.

In [ ]:
# Boxplots for skewed cat_cols

cat_cols_to_check = ["Street", "Utilities", "Condition2", "RoofMatl", "Heating"]

for col in cat_cols_to_check:
    sns.boxplot(data = df, x = col, y = "SalePrice")
    plt.show()

In [ ]:
# Utilities value counts

df["Utilities"].value_counts()

Drop utilities as clearly too imbalance.

In [ ]:
# Drop utilities col
cols_to_drop.append("Utilities")

**Model building**

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [ ]:
# Split data for training and convert cat variables, log transform sale price

X_train = pd.get_dummies(df.drop(columns=["SalePrice"] + cols_to_drop))

y_train = np.log(df["SalePrice"])

In [ ]:
# Initiate model and fit

rf = RandomForestRegressor()

rf.fit(X_train, y_train)

In [ ]:
# Load test data, does not include SalePrice

test = pd.read_csv("test.csv", index_col = "Id")

In [ ]:
# Pre process test data